# ARC-AGI-3 Team Workspace
**Reinforcement Learning (COMS4061A / COMS7071A), Assignment 2026**
**Team:** Elias Sepuru (1427726), Paballo (3134698), Banele (Benny) Mkhize (2663814)
**Due on Moodle:** Tuesday 27 October 2026, 23:59

---

# Part 0: Start here. What are we actually doing?

Give this part 30 minutes before you touch any project code. It tells the story of the assignment
from the beginning, and it has small cells you can run to *feel* the problem rather than just read
about it.

**About the cells in Part 0:** every code cell here is a **demo cell**. Demo cells are completely
independent of the project. They define their own little games, use only `numpy` and `matplotlib`,
and nothing later in the notebook depends on them. Run them, look at what they show, then **clear
their output** (hover over the output and click the clear icon, or use *Edit > Clear cell output*).
Skipping them breaks nothing. Running them twice breaks nothing.

## 0.1 Where we are coming from

Think back over this course. Every week added one piece to the same picture:

* **An agent and an environment.** The agent sees a **state**, picks an **action**, and the
  environment answers with a **reward** and the next state. That loop, repeated, is reinforcement
  learning.
* **Bandits** taught us the dilemma at the heart of it: **explore** (try something new to learn about
  it) or **exploit** (use what already seems best).
* **Markov decision processes** gave the loop a precise mathematical form: states, actions, transition
  probabilities, rewards, and a discount factor.
* **Dynamic programming** solved those problems when we already *knew* the rules of the environment.
* **Model-free learning** (Monte Carlo, SARSA, Q-learning) learned good behaviour by trial and error,
  *without* knowing the rules, by keeping a table of how good each action is in each state.
* **Function approximation** replaced the table with a neural network, so the agent can handle more
  states than any table could hold. In Lab 3 we used this to build a **Deep Q-Network** that learned to
  play an Atari game from the pixels on the screen.

In every one of those settings, two things were always handed to us:
1. **a reward signal** that told the agent, step by step, how well it was doing, and
2. **a clear task**: we always knew what "winning" meant.

**This assignment takes both away.** That is what makes it interesting, and that is what the rest of
Part 0 is about.

## 0.2 What is ARC-AGI?

**ARC** stands for the *Abstraction and Reasoning Corpus*. It was created in 2019 by François Chollet,
a researcher at Google, to test something most benchmarks do not: not *how good* a system is at a task,
but **how quickly it learns a task it has never seen before**. His argument was that any skill can be
bought with enough data and computing power, so skill alone is not intelligence. Intelligence is the
*efficiency* of turning a little experience into a new skill. **AGI** stands for *artificial general
intelligence*, the long-term goal the benchmark is meant to measure progress towards.

It has gone through three versions:

| Version | Year | What it is |
|---|---|---|
| ARC-AGI-1 | 2019 | Small coloured grid puzzles. You see two or three examples of a picture being transformed, then you must apply the same hidden rule to a new picture. Humans find most of them easy. For years, computers scored close to zero. |
| ARC-AGI-2 | 2025 | Harder puzzles of the same kind, and the cost of solving them is now reported too, so you cannot win just by spending more computing power. |
| **ARC-AGI-3** | 2025 preview, **2026 full release** | **Not puzzles any more: games.** You are dropped into a small interactive grid game with **no instructions, no stated goal, no explanation of the buttons, and no reward**, only a counter that goes up when you complete a level. You must work out what is going on by pressing buttons and watching what changes. |

**ARC Prize 2026** is the open competition built on ARC-AGI-3, run on **Kaggle** (a website that hosts
data science competitions). Our assignment asks us to enter it as a team, in two parts: the main
competition (our agent plays hidden games and gets a score) and the *Paper Track* (a short write-up).
The course marks our *investigation*, not our rank in the world.

**Why ARC-AGI-3 is a reinforcement learning problem:** an agent in an unknown environment, choosing
actions, learning from the consequences. It is exactly the loop from 0.1, with the safety rails removed.

## 0.3 What the agent sees

Every game screen is a grid of up to 64 by 64 cells, and every cell holds a number from 0 to 15, which
stands for one of 16 colours. That grid of numbers is **all** the agent gets. Run the cell below: it shows
the same made-up screen twice, once as the agent receives it, once as a human sees it.

In [ ]:
# DEMO CELL: independent of the project. Safe to run, safe to skip, nothing else uses it.
# When you have looked at the result, clear this cell's output (hover the output, click the
# small 'clear' icon, or Edit > Clear cell output) so it does not clutter the notebook.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# ARC-AGI-3 games use 16 colours, numbered 0 to 15. This is a made-up 12 by 12 screen.
demo_palette = ListedColormap(["#000000", "#1E93FF", "#F93C31", "#4FCC30", "#FFDC00", "#999999",
                               "#E53AA3", "#FF851B", "#87D8F1", "#921231", "#FFFFFF", "#555555",
                               "#2ECC40", "#B10DC9", "#7FDBFF", "#AAAAAA"])
demo_screen = np.zeros((12, 12), dtype=int)
demo_screen[0, :] = demo_screen[-1, :] = demo_screen[:, 0] = demo_screen[:, -1] = 5   # grey walls
demo_screen[9, 2] = 3      # a green block
demo_screen[2, 9] = 2      # a red square
demo_screen[5, 6] = 4      # a yellow thing
demo_screen[4:8, 4] = 5    # a grey wall segment

print("What the agent actually receives: a grid of numbers.\n")
print(demo_screen)
plt.figure(figsize=(4, 4))
plt.imshow(demo_screen, cmap=demo_palette, vmin=0, vmax=15)
plt.title("What a human sees when the same numbers are drawn as colours")
plt.xticks([]); plt.yticks([]); plt.show()
# Ask yourself: which object do YOU control? What is the goal? The agent is not told either.

## 0.4 Play a game with no instructions

Below is a tiny mystery game written in plain Python, in the spirit of ARC-AGI-3. It has two levels.
There are five actions, numbered 1 to 5. **Nobody will tell you what they do or what the goal is.**

1. Edit the line `demo_my_moves = [...]` to a list of actions, for example `[4, 4, 1]`.
2. Run the cell. You will see your last few screens and the number of levels completed.
3. Change the moves and run again. Keep going until you complete both levels.
4. Then try to do it in as **few actions** as possible.

While you play, **notice what you are doing in your head.** You will probably work out which square is
you, test what each button does, form a guess about what the goal is, check that guess, and carry
what you learned into level 2 when it looks different. Every one of those mental moves is something our
agent will have to do without a human brain. Write down what you noticed: it is the raw material for
the creative part of this project (section 0.9).

In [ ]:
# DEMO CELL: independent of the project. Safe to run, safe to skip, nothing else uses it.
# When you have looked at the result, clear this cell's output (hover the output, click the
# small 'clear' icon, or Edit > Clear cell output) so it does not clutter the notebook.
# DEMO CELL (independent). Clear its output when done.
# A tiny mystery game written in plain Python, in the spirit of ARC-AGI-3. There are no instructions.
# Actions: 1, 2, 3, 4, 5. You only see the screen and a "levels completed" counter.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

class DemoMysteryGame:
    """6 by 6 grid. Rules are deliberately NOT written here: discover them by playing."""
    LAYOUTS = [  # (player start, key position, door position) per level
        ((5, 0), (2, 1), (0, 5)),
        ((0, 0), (5, 4), (3, 0)),
    ]

    def __init__(self):
        self.level = 0
        self.levels_completed = 0
        self._load_level()

    def _load_level(self):
        start, self.key, self.door = self.LAYOUTS[self.level]
        self.player = list(start)
        self.has_key = False

    def screen(self):
        grid = np.zeros((6, 6), dtype=int)
        grid[self.door] = 2                       # red
        if not self.has_key:
            grid[self.key] = 4                    # yellow
        grid[tuple(self.player)] = 3              # green
        return grid

    def step(self, action):
        moves = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}
        if action in moves:
            row = min(5, max(0, self.player[0] + moves[action][0]))
            column = min(5, max(0, self.player[1] + moves[action][1]))
            self.player = [row, column]
        if tuple(self.player) == self.key:
            self.has_key = True
        if tuple(self.player) == self.door and self.has_key:
            self.levels_completed += 1
            if self.level + 1 < len(self.LAYOUTS):
                self.level += 1
                self._load_level()
        return self.screen(), self.levels_completed

    def won(self):
        return self.levels_completed == len(self.LAYOUTS)


# ======== EDIT THIS LINE, then run the cell. Try to complete both levels in as few actions as you can. ========
demo_my_moves = [4, 4, 1, 1, 1]
# ===============================================================================================================

demo_colours = ListedColormap(["#000000", "#1E93FF", "#F93C31", "#4FCC30", "#FFDC00"])
demo_game = DemoMysteryGame()
demo_frames = [demo_game.screen()]
for demo_action in demo_my_moves:
    demo_screen_now, demo_levels = demo_game.step(demo_action)
    demo_frames.append(demo_screen_now)
    if demo_game.won():
        break

demo_show = demo_frames[-8:]
figure, axes = plt.subplots(1, len(demo_show), figsize=(2 * len(demo_show), 2.3))
axes = np.atleast_1d(axes)
for axis, frame in zip(axes, demo_show):
    axis.imshow(frame, cmap=demo_colours, vmin=0, vmax=4); axis.set_xticks([]); axis.set_yticks([])
figure.suptitle("Your last moves, left to right")
plt.show()
print(f"Actions used: {len(demo_frames) - 1}   Levels completed: {demo_game.levels_completed} of 2   "
      f"{'YOU WON' if demo_game.won() else ''}")
# Stuck after a few tries? Remove the # on the next line for a hint.
# print("Hint: the green block is you. Something must be collected before the red square does anything.")

## 0.5 How the game is scored, and why random play fails

The official score for a level is
$$\text{score} = \left(\frac{\text{actions a human needed}}{\text{actions the agent needed}}\right)^2$$
(capped a little above 1). **Squared.** If a human needs 20 actions and the agent needs 200, the agent
scores $(20/200)^2 = 0.01$: one percent, *for winning*.

So finishing is only the entry ticket. The score rewards finishing **efficiently**. Run the cell below:
it lets a completely random agent play the mystery game 300 times.

In [ ]:
# DEMO CELL: independent of the project. Safe to run, safe to skip, nothing else uses it.
# When you have looked at the result, clear this cell's output (hover the output, click the
# small 'clear' icon, or Edit > Clear cell output) so it does not clutter the notebook.
# DEMO CELL (independent). Clear its output when done.
# A tiny mystery game written in plain Python, in the spirit of ARC-AGI-3. There are no instructions.
# Actions: 1, 2, 3, 4, 5. You only see the screen and a "levels completed" counter.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

class DemoMysteryGame:
    """6 by 6 grid. Rules are deliberately NOT written here: discover them by playing."""
    LAYOUTS = [  # (player start, key position, door position) per level
        ((5, 0), (2, 1), (0, 5)),
        ((0, 0), (5, 4), (3, 0)),
    ]

    def __init__(self):
        self.level = 0
        self.levels_completed = 0
        self._load_level()

    def _load_level(self):
        start, self.key, self.door = self.LAYOUTS[self.level]
        self.player = list(start)
        self.has_key = False

    def screen(self):
        grid = np.zeros((6, 6), dtype=int)
        grid[self.door] = 2                       # red
        if not self.has_key:
            grid[self.key] = 4                    # yellow
        grid[tuple(self.player)] = 3              # green
        return grid

    def step(self, action):
        moves = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}
        if action in moves:
            row = min(5, max(0, self.player[0] + moves[action][0]))
            column = min(5, max(0, self.player[1] + moves[action][1]))
            self.player = [row, column]
        if tuple(self.player) == self.key:
            self.has_key = True
        if tuple(self.player) == self.door and self.has_key:
            self.levels_completed += 1
            if self.level + 1 < len(self.LAYOUTS):
                self.level += 1
                self._load_level()
        return self.screen(), self.levels_completed

    def won(self):
        return self.levels_completed == len(self.LAYOUTS)


# DEMO C: how long does pure randomness take to win? And what score would that get?
demo_random = np.random.default_rng(0)
demo_actions_needed = []
for demo_try in range(300):
    demo_game = DemoMysteryGame()
    for demo_count in range(1, 20_001):
        demo_game.step(int(demo_random.integers(1, 6)))
        if demo_game.won():
            break
    demo_actions_needed.append(demo_count)

demo_human_actions = 16     # roughly what a person needs for both levels once they get the idea
demo_scores = [min((demo_human_actions / a) ** 2, 1.15) for a in demo_actions_needed]
print(f"Random agent: median {int(np.median(demo_actions_needed))} actions to win both levels "
      f"(a person needs about {demo_human_actions}).")
print(f"ARC-style efficiency score of the random agent: {np.mean(demo_scores):.4f} out of about 1.")

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.hist(demo_actions_needed, bins=40, color="tab:gray")
left.axvline(demo_human_actions, color="red", ls="--", label="a person")
left.set_xlabel("actions needed to win"); left.set_ylabel("number of random tries"); left.legend()
left.set_title("Random play: winning eventually is not the problem")
demo_agent_actions = np.arange(demo_human_actions, 400)
right.plot(demo_agent_actions, (demo_human_actions / demo_agent_actions) ** 2)
right.set_xlabel("actions the agent used"); right.set_ylabel("score for the level")
right.set_title("Score = (human actions / agent actions) squared")
plt.tight_layout(); plt.show()

**What that shows:** random play does eventually win this tiny game, but it needs hundreds of actions
where a person needs about sixteen, so its score is close to zero. Real ARC-AGI-3 games are much bigger,
so random play usually never wins at all.

And here is the deeper problem for reinforcement learning: **until the agent completes a level for the
first time, every single reward it has ever received is zero.** Q-learning, a Deep Q-Network, Proximal
Policy Optimisation: all of them learn from reward. With nothing but zeros, they learn nothing. The agent
is not learning slowly; it is not learning at all, until it stumbles on its first success.

## 0.6 Exploration on purpose

If reward is missing, the agent needs a different reason to try new things. Here is the simplest one,
straight from the Bandits lecture: **optimistic initial values.** The agent *assumes* every action it has
never tried is excellent. Each time it tries one and it turns out to be ordinary, that belief comes down.
The effect: it keeps being drawn towards what it has not tried yet, so it explores systematically
instead of wandering at random.

The cell below takes the course's ordinary **tabular Q-learning** and runs it on a bigger, 12 by 12
version of the key-and-door game, two ways: with ordinary epsilon-greedy exploration (mostly greedy,
random 10% of the time), and with optimistic initial values. Same algorithm, same game, same budget.
It takes about half a minute.

In [ ]:
# DEMO CELL: independent of the project. Safe to run, safe to skip, nothing else uses it.
# When you have looked at the result, clear this cell's output (hover the output, click the
# small 'clear' icon, or Edit > Clear cell output) so it does not clutter the notebook.

import numpy as np
import matplotlib.pyplot as plt

class DemoBigRoom:
    """12 by 12 room. Start bottom-left. A key in the top-left corner. A door in the top-right corner.
    The door only completes the level once the key has been collected."""
    def __init__(self, size=12):
        self.size, self.player, self.has_key, self.levels_completed = size, [size - 1, 0], False, 0
    def step(self, action):
        moves = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}
        if action in moves:
            self.player = [min(self.size - 1, max(0, self.player[0] + moves[action][0])),
                           min(self.size - 1, max(0, self.player[1] + moves[action][1]))]
        if tuple(self.player) == (0, 0):
            self.has_key = True
        if tuple(self.player) == (0, self.size - 1) and self.has_key:
            self.levels_completed = 1
    def won(self):
        return self.levels_completed == 1

def demo_q_learning(optimistic, seed, episodes=40, action_cap=1500):
    """Tabular Q-learning from the course. Reward is +1 ONLY when the level is completed."""
    rng = np.random.default_rng(seed)
    starting_value = 1.0 if optimistic else 0.0       # the whole difference between the two agents
    q_table, actions_per_episode = {}, []
    for _ in range(episodes):
        room = DemoBigRoom()
        state = (*room.player, room.has_key)
        for step in range(1, action_cap + 1):
            q_values = q_table.setdefault(state, np.full(5, starting_value))
            if not optimistic and rng.random() < 0.1:
                action = int(rng.integers(5))                                   # epsilon-greedy: random 10%
            else:
                action = int(rng.choice(np.flatnonzero(q_values == q_values.max())))   # greedy
            room.step(action + 1)
            next_state = (*room.player, room.has_key)
            reward = room.levels_completed
            next_values = q_table.setdefault(next_state, np.full(5, starting_value))
            target = reward + (0.0 if room.won() else 0.95 * next_values.max())
            q_values[action] += 0.5 * (target - q_values[action])
            state = next_state
            if room.won():
                break
        actions_per_episode.append(step)
    return np.array(actions_per_episode)

plt.figure(figsize=(8, 4))
for optimistic, colour, name in [(False, "tab:gray", "epsilon-greedy (explores by chance)"),
                                 (True, "tab:purple", "optimistic (explores on purpose)")]:
    runs = np.array([demo_q_learning(optimistic, seed) for seed in range(5)])
    never_found = int((runs == 1500).sum())
    plt.plot(runs.mean(axis=0), color=colour, label=name)
    plt.fill_between(range(runs.shape[1]), runs.min(axis=0), runs.max(axis=0), color=colour, alpha=0.15)
    print(f"{name}: episodes that never found the exit = {never_found} of {runs.size}; "
          f"average actions in the last 10 episodes = {runs[:, -10:].mean():.0f}")
plt.xlabel("episode"); plt.ylabel("actions needed (lower is better)")
plt.title("Same algorithm, same game: exploring on purpose changes everything")
plt.legend(); plt.show()

**What that shows:** with exploration left to chance, the agent often never finds the exit at all in a
bigger room. With a deliberate reason to explore, it finds it far more often and learns a shorter route.

Our project's **Improvement 2 (the novelty bonus)** is the same idea, scaled up. In ARC-AGI-3 there are
far too many possible screens to keep a table, so instead of an optimistic table we pay the agent a small
extra reward for reaching **screens it has rarely seen before**.

**An honest warning, worth knowing before Task 5:** a curiosity bonus done carelessly can backfire. When
we first tried a reward bonus in this very demo, the agent learned to circle the area it already knew,
because that is where the bonus it had collected was. Whether a bonus helps is a question to **measure**,
not assume. That measuring is exactly what the assignment gives marks for.

## 0.7 Why the agent needs memory

Some games hide information. You picked up a key, but nothing on screen shows it. You pressed a switch
somewhere else. A light flashed at the start and is gone now. Two moments can look **pixel for pixel
identical** and still need different actions. An agent that only looks at the current screen cannot tell
them apart, no matter how well it is trained. Run the cell below.

In [ ]:
# DEMO CELL: independent of the project. Safe to run, safe to skip, nothing else uses it.
# When you have looked at the result, clear this cell's output (hover the output, click the
# small 'clear' icon, or Edit > Clear cell output) so it does not clutter the notebook.
# A corridor game: at the start a light shows LEFT or RIGHT. Then you walk down a grey corridor
# (every screen looks identical) to a junction, and must turn the way the light said.
import numpy as np
demo_rng = np.random.default_rng(0)
demo_episodes = 10_000
demo_correct_way = demo_rng.choice(["LEFT", "RIGHT"], size=demo_episodes)

# Agent 1 only sees the CURRENT screen. At the junction every screen is the same, so the best it can
# ever do is pick one way and always go that way.
demo_no_memory_success = np.mean(demo_correct_way == "LEFT")
# Agent 2 keeps the first screen in memory, so at the junction it still knows what the light said.
demo_memory_success = np.mean(demo_correct_way == demo_correct_way)

print(f"Best possible agent WITHOUT memory: {demo_no_memory_success:.0%} success, no matter how long it trains.")
print(f"Agent WITH one remembered screen : {demo_memory_success:.0%} success.")
print("\nThe ceiling is not a training problem. It is missing information. Many ARC-AGI-3 games hide")
print("state like this (which key you hold, which switch you already pressed). That is Improvement 1.")

**What that shows:** that 50% ceiling is not a training failure, it is missing information. The fix is
memory: let the agent see more than the current screen. Our project's **Improvement 1** does this two
ways. **Frame stacking** shows the agent the last few screens. **Adaptive Stacking** (a method published by
our lecturer, Geraud Nangue Tasse) goes further: the agent gets a fixed number of memory slots and
**learns which screens are worth keeping**, instead of always keeping just the most recent ones.

In technical terms, the game is *partially observable*: the screen is only part of the true state.
You will see the phrase "partially observable Markov decision process" in the reading; this is what it means.

## 0.8 What we should have at the end, and what "success" looks like

**The deliverable**, in plain words:
* two learning agents: a **Deep Q-Network** (an algorithm from the course) and **Proximal Policy
  Optimisation** (an algorithm not covered in the course, which learns a policy directly);
* a **random agent** as the sanity check, the floor that everything should beat;
* for each learning agent, a **baseline** and **two improvements** (memory, then exploration), so six
  learning versions in total, each trained and evaluated with **three different random seeds**;
* an honest comparison, on games the agents were **never trained on**;
* an 8-page report, a Kaggle submission, and a short Paper Track write-up.

**What success honestly looks like.** ARC-AGI-3 is hard for everyone, including big research labs. Our
agents will probably score close to zero on the official measure, and **that is fine**: the brief says
in plain words that strong marks do not need a world-leading score. Ninety of the hundred marks are for
the investigation. So the signals we are actually looking for are smaller and measurable:
* Does an improved version **complete its first level sooner**, or complete more levels, than the version before it?
* Does it **waste fewer actions** (for example, fewer actions the game does not even offer)?
* Does it **see more different screens** during training (a sign that exploration is working)?
* Is the difference **bigger than the spread between seeds**, or could it just be luck?

For every change we make, the report tells a four-part story: **the limitation we observed, our
hypothesis, what we changed, and what the results showed.** A change that did *not* help still earns marks
if that story is told well.

## 0.9 Building intuition, and where creativity comes in

The two improvements are already chosen, but *how* we build them, and what we try beyond them, is where
our own ideas matter. A practical way to get ideas:

1. **Play real ARC-AGI-3 games yourself** at [arcprize.org](https://arcprize.org) (they run in the browser).
   Play two or three. Do not look anything up.
2. **Write down every trick you used to figure each game out.** Be specific: "I clicked the thing that
   looked different", "I remembered the colour I saw at the start", "I stopped pressing a button once it
   did nothing twice".
3. **Turn each trick into something an agent could do.** For example:

| What you did as a human | What it could become for the agent |
|---|---|
| Ignored buttons that did nothing | Learn, per game, which actions change the screen, and prefer those |
| Clicked on objects, not on empty background | Click targets placed on objects instead of on a fixed 8 by 8 grid |
| Remembered something from earlier | Memory (Improvement 1), and *which* things are worth remembering (Adaptive Stacking) |
| Tried things you had not tried yet | Novelty bonus (Improvement 2), perhaps counted over objects rather than whole screens |
| Noticed level 2 used the same rule as level 1 | Keep what was learned across levels instead of starting fresh |

4. **Post your best idea in the group chat as a one-line hypothesis:** "If the agent does X, it will
   complete its first level sooner, because Y." The good ones become our ablations or extra experiments.

That habit, *observe a limitation, form a hypothesis, test it, report it honestly*, is the whole
assignment in one sentence. It is also exactly what research is.

---
# Part 1: How this notebook works

| Part | What it is | Who edits it |
|---|---|---|
| **Part 0: Start here** | The story of the assignment, with demo cells to run. | Nobody. Read and run. |
| **Part 1: How this notebook works** | This part: the rules, the decisions already made, who does what, the dates. | Nobody. Read only. |
| **Part 2: Shared setup** | Installs the software, sets the shared settings, prepares the games. | Benny only. Everyone runs it. |
| **Part 3: Workspace, Round 1** | Building the two basic agents and the evaluation. One section per person. | Only the named owner. |
| **Part 4: Workspace, Round 2** | The improvements, the experiments, the figures, the Kaggle agent. | Only the named owner. |
| **Part 5: Review log** | Who is done, what was reviewed, what goes into the final file. | Benny. |

**Everything from Part 3 onwards is the workspace: that is where the real code gets written.**
When every section is finished and reviewed, Benny combines them into a single Python file,
`arc_project.py`. That file is what we run for training and experiments, and it is the source of the
Kaggle submission and the Moodle ZIP.

## How work moves (the cycle)

1. **Open your section.** Use the table of contents on the left of Colab and look for your name.
2. **Read your task card**, the text cell at the top of your section. It says what to build, the exact
   function names with their inputs and outputs, why it matters for the marks, and a **blocker flag**
   saying what it waits on and who waits on it.
3. **Write the code** in the code cell. Every `TODO` comment says what goes there. Keep the function
   names exactly as they are: other people's code calls them by those names.
4. **Run your self-check cell.** It prints `PASS`, or tells you what is still wrong.
5. **When it prints PASS**, post *"Task N done, self-check PASS"* in the group chat.
6. **Benny reviews it.** It is either approved and goes into `arc_project.py`, or it comes back with notes
   in the review log (Part 5) and you go round again.
7. Repeat until every row of the review log says **APPROVED**.

## Rules that keep four people from breaking each other's work

* **Work in your own copy** (*File > Save a copy in Drive*) and only paste your finished section back.
  Colab does not merge edits: if two people edit the same file at once, one overwrites the other.
* **Edit only your own section.** If you need a change in someone else's, message them.
* **Do not change a function's name, inputs or outputs** (we call this the *contract*). If you think a
  contract is wrong, say so in the chat first.
* **Comment as you write.** Assume the reader is tired and has never seen your code. The report is
  written from these comments.
* **Write down what did not work** in the notes cell at the end of your section. The brief gives marks
  for honest analysis of failed ideas.
* **Never paste a password or an access key into the notebook.** Offline play needs none.
* **Declare any help from AI tools** in your notes cell. The course requires it, and it goes into the
  report's contribution statement.

## The decisions already made

These were chosen so we can start immediately. Do not reopen them without a strong reason.

| Decision | Our choice | Why, in plain words |
|---|---|---|
| The algorithm from the course | **Deep Q-Network**, using the ready-made version in the Stable-Baselines3 library (`DQN` in the code) | We built one in Lab 3, so we can explain it well. The library version is quicker and less error-prone than writing it again. |
| The algorithm not from the course | **Proximal Policy Optimisation with action masking**, from the sb3-contrib library (`MaskablePPO` in the code) | It learns a policy directly. "Masking" means it is never allowed to pick an action the current game does not offer, so it wastes nothing. Benny has trained it extensively. |
| Improvement 1, for both agents | **Memory**: first frame stacking, then Adaptive Stacking | Many games hide information (section 0.7). |
| Improvement 2, for both agents | **Exploration**: a novelty bonus for rarely seen screens | Without it, the agent learns nothing until its first lucky success (sections 0.5 and 0.6). |
| Reward | **+1 every time the levels-completed counter goes up.** Nothing else, apart from the novelty bonus during training. | This is the honest minimum. Any extra reward we invent is only for training and is **never** reported as the official score. |
| What the agent sees | The latest screen, padded to 64 by 64, stored as a one-channel image | So the standard image network in the library can read it directly. |
| What the agent can do | **70 actions**: the five simple actions, the undo action, and clicks on the centres of an 8 by 8 grid of blocks (64 click positions) | A click can land on any of 4,096 cells. That is far too many to explore, so we reduce it. We report this as a design decision. |
| Which games | The public games are split: **every fourth game is held out** and never used for training | The brief requires us to test on games the agent has not seen, and to list every game we used. |

## The versions we will evaluate

Every results file uses these exact names.

| Version name | Agent | What is different |
|---|---|---|
| `random_v0` | random | The sanity check: picks any allowed action at random. |
| `dqn_v0_baseline` and `ppo_v0_baseline` | Deep Q-Network and Proximal Policy Optimisation | The plain agent on the shared game setup. |
| `dqn_v1_memory` and `ppo_v1_memory` | the same | Plus memory (frame stacking or Adaptive Stacking, whichever works better on the training games). |
| `dqn_v2_memory_novelty` and `ppo_v2_memory_novelty` | the same | Plus the novelty bonus, on top of memory. |
| `dqn_v2_novelty_only` and `ppo_v2_novelty_only` | the same | Novelty bonus *without* memory. This is the **ablation** the brief asks for: it separates the effect of the two improvements. |

## Who does what

| Round | Elias | Benny | Paballo | Benny |
|---|---|---|---|---|
| **0 (tonight)** | Task 0: check what a real game returns | Join both Kaggle competitions, set up our team | Read Part 0 and the primer | Set up the shared Drive folder |
| **1: the basic agents** | **Task 1**: the game adapter. Everyone's real runs wait on this one. | **Task 2**: random agent, evaluation rules, results file | **Task 3**: Deep Q-Network baseline | **Task 4**: Proximal Policy Optimisation baseline |
| **2: improvements** | **Task 7**: experiment runner (all versions, all seeds) | **Task 8**: figures, tables, Kaggle agent | **Task 5**: novelty bonus | **Task 6**: memory (frame stacking and Adaptive Stacking) |
| **3: the report** | Methods, and the game set-up as a Markov decision process | Results figures, reproducibility | The two algorithms explained | Introduction, analysis, putting it together |

Tasks 2, 3 and 4 **do not wait for Elias**: Part 2 includes a stand-in game with exactly the same shape
as the real one, so everyone can build and test straight away.

## Timeline

| Date | Milestone |
|---|---|
| Wed 14 Oct | Round 0 done. Everyone has read Part 0. |
| Mon 19 Oct | All Round 1 self-checks print PASS. Random agent evaluated. |
| Tue 20 Oct | First Kaggle submission, even if it is the random agent, so we know the submission process works. |
| Thu 22 Oct | Both basic agents evaluated over 3 seeds. Round 2 starts. |
| Sat 24 Oct | All Round 2 self-checks print PASS. All versions training. |
| Mon 26 Oct | All results in. Figures made. Report drafted. |
| Tue 27 Oct | Report final, Kaggle submission chosen, ZIP on Moodle by 23:59. |

---
# Part 2: Shared setup (Benny edits; everyone runs)

Run these cells top to bottom every time you open the notebook. They install packages, set
shared settings, and define a **stand-in environment** (`FakeArcEnv`) that behaves like the real
adapter, so Tasks 2, 3 and 4 can be built and tested before Task 1 is finished.

In [ ]:
# ==============================================================================
# BLOCKER FOR EVERYONE. Owner:Benny. Blocked by: nothing.
# Blocks: every cell below. Nobody can run anything until this installs.
# ==============================================================================
# --- Installs (Colab already has torch, numpy, pandas, matplotlib) ---
# arc-agi        : the official ARC-AGI-3 toolkit (runs games locally, about 2000 frames per second)
# stable-baselines3 / sb3-contrib : DQN, PPO and MaskablePPO implementations
# gymnasium      : the standard environment interface SB3 expects
!pip -q install arc-agi "stable-baselines3>=2.3" "sb3-contrib>=2.3" "gymnasium>=0.29"

In [ ]:
# ==============================================================================
# BLOCKER FOR EVERYONE. Owner: Benny. Blocked by: the install cell above.
# Blocks: every task (shared settings, the results folder, set_global_seed).
# ==============================================================================
# --- Imports used across the whole notebook ---
import os, time, random, hashlib, json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
import torch

# --- Shared settings. Change these ONLY in this cell, so every task agrees. ---
SEEDS = [0, 1, 2]                  # the brief asks for at least 3 independent seeds
MAX_STEPS_PER_EPISODE = 500        # our own action cap per episode (this is TRUNCATION, not game over)
COARSE_GRID = 8                    # ACTION6 click targets = centres of an 8x8 grid of 8x8-cell blocks
N_SIMPLE_ACTIONS = 6               # ACTION1..ACTION5 and ACTION7
N_ACTIONS = N_SIMPLE_ACTIONS + COARSE_GRID * COARSE_GRID   # = 70
GRID_SIZE = 64                     # frames are at most 64x64; smaller frames are padded
TRAIN_TIMESTEPS_SMOKE = 2_000      # tiny runs used by self-checks
TRAIN_TIMESTEPS_REAL = 50_000      # per seed per version for the real experiments (raise if time allows)

# Where results and models are saved. On Colab, mount Drive so nothing is lost when the session dies.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ARC_RL_Project")
except ImportError:
    PROJECT_DIR = Path("./ARC_RL_Project")
RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"
for folder in (RESULTS_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "evaluation_results.csv"


def set_global_seed(seed: int) -> None:
    """Seed every random number generator we use, so a run can be repeated exactly."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


print(f"Actions per step: {N_ACTIONS}. Results go to: {RESULTS_DIR}")

In [ ]:
# ==============================================================================
# BLOCKER. Owner: Benny. Blocked by: the two cells above.
# Blocks: Task 0 (Elias), Task 1 (Elias), Task 7 (Elias) and any evaluation on real games
# (Benny, Task 2), because they need TRAINING_GAME_IDS / HELD_OUT_GAME_IDS.
# Does NOT block Tasks 2, 3, 4, 5, 6 while they are being built on FakeArcEnv.
# ==============================================================================
# --- The game list and the training / held-out split (computed once, shared by everyone) ---
# OFFLINE mode: the first run downloads the public games into ./environment_files, after that no
# internet is needed. This matches Kaggle, where internet is switched off during evaluation.
from arc_agi import Arcade, OperationMode
from arcengine import GameAction

ARCADE = Arcade(operation_mode=OperationMode.OFFLINE)   # NORMAL = local + remote; switch to OFFLINE once games are cached
ALL_GAME_IDS = sorted(info.game_id for info in ARCADE.get_environments())
HELD_OUT_GAME_IDS = ALL_GAME_IDS[3::4]                  # every 4th game, never trained on
TRAINING_GAME_IDS = [g for g in ALL_GAME_IDS if g not in HELD_OUT_GAME_IDS]

print(f"{len(ALL_GAME_IDS)} public games")
print("Training games :", TRAINING_GAME_IDS)
print("Held-out games :", HELD_OUT_GAME_IDS)
# This exact list goes into the report (the brief: "Disclose every environment used during development").

In [ ]:
# ==============================================================================
# UNBLOCKER. Owner: Benny. Blocked by: the settings cell.
# This stand-in is what lets Benny (Task 2), Paballo (Tasks 3 and 5) and Benny (Tasks 4 and 6)
# start WITHOUT waiting for Elias's Task 1. Run it before any self-check.
# ==============================================================================
# --- Stand-in environment so Tasks 2, 3 and 4 can be built before Task 1 is finished ---
# It has EXACTLY the same contract as the real ArcGymEnv (same spaces, same info keys,
# same action_masks method). The "game": a hidden target cell; clicking the right coarse block
# completes a level. It is deliberately easy so you can see learning happen in a smoke test.

class FakeArcEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, game_ids=None, max_steps=MAX_STEPS_PER_EPISODE, seed=0):
        super().__init__()
        self.observation_space = spaces.Box(0, 255, shape=(1, GRID_SIZE, GRID_SIZE), dtype=np.uint8)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self.max_steps = max_steps
        self.random_generator = np.random.default_rng(seed)
        self.game_ids = game_ids or ["fake-0001"]

    def _observation(self):
        grid = np.zeros((1, GRID_SIZE, GRID_SIZE), dtype=np.uint8)
        block_row, block_col = divmod(self.target_block, COARSE_GRID)
        grid[0, block_row * 8:(block_row + 1) * 8, block_col * 8:(block_col + 1) * 8] = 3 * 16
        return grid

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.target_block = int(self.random_generator.integers(COARSE_GRID * COARSE_GRID))
        self.steps_taken = 0
        self.levels_completed = 0
        return self._observation(), self._info(valid=True)

    def step(self, action):
        self.steps_taken += 1
        action = int(action)
        reward = 0.0
        if action == N_SIMPLE_ACTIONS + self.target_block:
            reward = 1.0
            self.levels_completed += 1
            self.target_block = int(self.random_generator.integers(COARSE_GRID * COARSE_GRID))
        terminated = self.levels_completed >= 3                 # "WIN" after 3 levels
        truncated = self.steps_taken >= self.max_steps
        return self._observation(), reward, terminated, truncated, self._info(valid=True)

    def action_masks(self):
        return np.ones(N_ACTIONS, dtype=bool)

    def _info(self, valid):
        return {"game_id": self.game_ids[0], "levels_completed": self.levels_completed,
                "win_levels": 3, "state": "WIN" if self.levels_completed >= 3 else "NOT_FINISHED",
                "actions_used": self.steps_taken, "action_was_valid": valid}


print("FakeArcEnv ready. Use it until ArcGymEnv (Task 1) passes its self-check.")

---
# Part 3: Workspace, Round 1 (baselines)

**From here down is the workspace.** Each section belongs to one person.

## Task 0 (Elias, 15 minutes, tonight): confirm what a game actually returns

> **BLOCKER FLAG:** BLOCKER. Blocked by: shared setup. **Blocks: Task 1 (Elias)**, because the adapter needs the real field names this prints.


The documentation does not list every field of the object `env.reset()` and `env.step()` return.
Our adapter depends on the exact names, so we check them on a real game before writing anything.

**Do:** run the cell below and paste its whole output into the group chat.

In [ ]:
# ==============================================================================
# BLOCKER. Owner: Elias. Blocked by: shared setup (game list cell).
# Blocks: Task 1 (Elias's own adapter needs these exact field names). Do this first, tonight.
# ==============================================================================
# OWNER: Elias.  Just run it.
probe_env = ARCADE.make(TRAINING_GAME_IDS[0], seed=0)
first_frame = probe_env.reset()
print("Type returned by reset():", type(first_frame))
print("Public fields:", [name for name in dir(first_frame) if not name.startswith("_")])
for field in ["state", "levels_completed", "win_levels", "available_actions", "frame", "game_id", "guid"]:
    value = getattr(first_frame, field, "<missing>")
    if field == "frame" and value != "<missing>":
        print("frame: list of", len(value), "grid(s); first grid shape:", np.array(value[0]).shape)
    else:
        print(f"{field}: {value}")
print("env.action_space:", probe_env.action_space)
after_step = probe_env.step(probe_env.action_space[0])
print("After one step, state:", getattr(after_step, "state", None),
      " levels_completed:", getattr(after_step, "levels_completed", None))

## Task 1 (Elias): the environment adapter `ArcGymEnv`  *(critical path, do first)*

> **BLOCKER FLAG:** CRITICAL BLOCKER. Blocked by: Task 0. **Blocks: anything on real games**: Benny's real evaluation (Task 2), Paballo's and Benny's real training runs (Tasks 3, 4), the runner (Task 7), and Benny's Kaggle agent (Task 8). Tasks 2 to 6 can still be BUILT on FakeArcEnv meanwhile.


**Why this matters:** the official interface is not a Gymnasium environment, and SB3 only speaks
Gymnasium. This class is the bridge, and it **is** our MDP definition: the report's first rubric
line (10 marks) is written straight from it.

**You must deliver** a class `ArcGymEnv(gym.Env)` with exactly this contract:

| Member | Contract |
|---|---|
| `__init__(self, game_ids, max_steps=MAX_STEPS_PER_EPISODE, seed=0)` | `game_ids` is a list; one game is picked at random each `reset`. |
| `observation_space` | `Box(0, 255, shape=(1, 64, 64), dtype=uint8)` |
| `action_space` | `Discrete(70)` |
| `reset(seed=None, options=None)` | returns `(observation, info)` |
| `step(action)` | returns `(observation, reward, terminated, truncated, info)` |
| `action_masks()` | returns a `bool` array of length 70, `True` = this game offers that action |
| `info` keys | `game_id, levels_completed, win_levels, state, actions_used, action_was_valid` |

**Rules for the semantics** (write a comment next to each, they go in the report):
* **Observation:** last grid of the returned frame list, padded with zeros to 64 by 64, multiplied by 16.
* **Action index mapping:** 0-4 are ACTION1-5, 5 is ACTION7, 6 to 69 are ACTION6 with
  `x = 8*column + 4`, `y = 8*row + 4` where `row, column = divmod(index - 6, 8)`.
* **Invalid action** (not in the game's available actions): do **not** send it to the game; return the
  same observation, reward 0, `action_was_valid=False`. (DQN cannot mask, so it will sometimes try these.)
* **Reward:** `levels_completed` now minus `levels_completed` before the step. Nothing else.
* **terminated** = state is `WIN` or `GAME_OVER`. **truncated** = we hit `max_steps`. These are
  different on purpose: getting them mixed up breaks value learning.

**Done when:** the self-check below prints `PASS`.

In [ ]:
# ==============================================================================
# CRITICAL BLOCKER. Owner: Elias. Blocked by: Task 0.
# Blocks (on real games): Task 2 real evaluation (Benny), Task 3 real DQN training (Paballo),
# Task 4 real PPO training (Benny), Task 7 runner (Elias), Task 8 Kaggle agent (Benny, must copy
# frame_to_observation exactly). Highest priority cell in the notebook.
# ==============================================================================
# OWNER: Elias.  Only Elias edits this cell.

class ArcGymEnv(gym.Env):
    """Gymnasium adapter around one ARC-AGI-3 game (chosen at random from game_ids on every reset)."""

    metadata = {"render_modes": []}

    def __init__(self, game_ids, max_steps=MAX_STEPS_PER_EPISODE, seed=0):
        super().__init__()
        self.game_ids = list(game_ids)
        self.max_steps = max_steps
        self.random_generator = np.random.default_rng(seed)
        self.observation_space = spaces.Box(0, 255, shape=(1, GRID_SIZE, GRID_SIZE), dtype=np.uint8)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self.game_environment = None       # the official EnvironmentWrapper for the current game
        self.latest_frame = None            # the last object the official env returned
        self.current_game_id = None
        self.steps_taken = 0
        self.last_observation = np.zeros((1, GRID_SIZE, GRID_SIZE), dtype=np.uint8)   # fallback screen


    # ---------- helpers ----------

    def frame_to_observation(self, frame_object):
        """Turn the official frame object into a (1, 64, 64) uint8 array."""
        if not frame_object.frame:              # some steps return no grid at all (e.g. after game over)
            return self.last_observation        # keep showing the last screen we had
        last_grid = frame_object.frame[-1]
        grid_array = np.array(last_grid, dtype=np.int32)
        
        h, w = grid_array.shape
        pad_h = max(0, GRID_SIZE - h)
        pad_w = max(0, GRID_SIZE - w)
        padded_array = np.pad(grid_array, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0)
        
        scaled_array = (padded_array * 16).astype(np.uint8)
        observation = np.expand_dims(scaled_array, axis=0)
        self.last_observation = observation
        return observation

    def index_to_game_action(self, action_index):
        """Map our integer action (0..69) to (GameAction, data dict or None)."""
        simple_actions = [1, 2, 3, 4, 5, 7]
        
        if action_index < 6:
            return (simple_actions[action_index], None)
        else:
            row, column = divmod(action_index - 6, COARSE_GRID)    # which of the 8x8 blocks
            return (6, {"x": 8 * column + 4, "y": 8 * row + 4})    # click the centre of that block

    def action_masks(self):
        """Boolean array of length 70 (or N_ACTIONS): True where the CURRENT game offers that action."""
        mask = np.zeros(self.action_space.n, dtype=bool)
        available = getattr(self.latest_frame, "available_actions", [])
        
        simple_actions = [1, 2, 3, 4, 5, 7]
        for i, act in enumerate(simple_actions):
            if act in available:
                mask[i] = True
                
        if 6 in available:
            mask[6:] = True
            
        return mask

    def _info(self, action_was_valid):
        """Build the info dict (keys are fixed by the contract)."""
        return {
            "game_id": self.current_game_id,
            "levels_completed": int(getattr(self.latest_frame, "levels_completed", 0)),
            "win_levels": int(getattr(self.latest_frame, "win_levels", 0) or 0),
            "state": str(getattr(self.latest_frame, "state", "UNKNOWN")),
            "actions_used": self.steps_taken,
            "action_was_valid": action_was_valid,
        }

    # ---------- the Gymnasium API ----------

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.current_game_id = self.random_generator.choice(self.game_ids)
        self.game_environment = ARCADE.make(self.current_game_id, seed=seed)
        self.latest_frame = self.game_environment.reset()
        self.steps_taken = 0
        
        return (self.frame_to_observation(self.latest_frame), self._info(True))

    def step(self, action):
        self.steps_taken += 1
        
        if not self.action_masks()[action]:
            obs = self.frame_to_observation(self.latest_frame)
            truncated = self.steps_taken >= self.max_steps
            return (obs, 0.0, False, truncated, self._info(False))
            
        old_levels = int(getattr(self.latest_frame, "levels_completed", 0))
        game_action, data = self.index_to_game_action(action)
        
        if data is None:
            self.latest_frame = self.game_environment.step(game_action)
        else:
            self.latest_frame = self.game_environment.step(game_action, data=data)
            
        new_levels = int(getattr(self.latest_frame, "levels_completed", 0))
        reward = float(new_levels - old_levels)
        
        current_state = str(getattr(self.latest_frame, "state", "UNKNOWN"))
        terminated = ("WIN" in current_state) or ("GAME_OVER" in current_state)   # state looks like "GameState.WIN"
        truncated = (self.steps_taken >= self.max_steps) and not terminated
        
        return (self.frame_to_observation(self.latest_frame), reward, terminated, truncated, self._info(True))

In [ ]:
# SELF-CHECK for Task 1 (anyone may run; do not edit)
def self_check_task1():
    try:
        environment = ArcGymEnv(TRAINING_GAME_IDS[:2], max_steps=50, seed=0)
        observation, info = environment.reset(seed=0)
        assert observation.shape == (1, 64, 64) and observation.dtype == np.uint8, "observation shape/dtype wrong"
        assert set(info) == {"game_id", "levels_completed", "win_levels", "state", "actions_used", "action_was_valid"}, "info keys wrong"
        mask = environment.action_masks()
        assert mask.shape == (N_ACTIONS,) and mask.dtype == bool and mask.any(), "action_masks wrong"
        for _ in range(60):
            valid_indices = np.flatnonzero(environment.action_masks())
            observation, reward, terminated, truncated, info = environment.step(int(np.random.choice(valid_indices)))
            assert reward >= 0, "reward should never be negative in the baseline"
            if terminated or truncated:
                break
        assert truncated or terminated, "episode should end by max_steps=50"
        from stable_baselines3.common.env_checker import check_env
        check_env(ArcGymEnv(TRAINING_GAME_IDS[:1], max_steps=20), warn=True)
        print("Task 1 PASS")
    except NotImplementedError as missing:
        print("Task 1 NOT DONE YET:", missing)

self_check_task1()

In [ ]:
# NOTES FOR THE REPORT (Elias): what you decided, what you tried that failed, AI help used.
# -

## Task 2 (Benny): random reference policy, evaluation protocol, results file

> **BLOCKER FLAG:** BLOCKER. Blocked by: nothing to start (use FakeArcEnv). **Blocks: Task 7 runner (Elias)**, which calls evaluate and save_results, and **Task 8 figures (Benny)**, which read the results file. The protocol must be frozen before Round 2 is evaluated.


**Why this matters:** "Baseline implementation and evaluation" is 20 marks, and the brief says the
evaluation protocol must be fixed **before** any improvement. Every other task will call your
`evaluate` function, so every number in the report comes through your code.

**You must deliver:**

| Function | Contract |
|---|---|
| `random_policy(observation, info, action_mask) -> int` | uniform over the actions where `action_mask` is True |
| `evaluate(policy_function, version, algorithm, seed, game_ids, episodes_per_game=1, max_steps=MAX_STEPS_PER_EPISODE, make_environment=None, timesteps_at_checkpoint=None, checkpoint_path=None) -> pd.DataFrame` | plays each game, one row per (game, episode) |
| `save_results(results_table)` | appends to `RESULTS_CSV` without duplicating the header |

**Row columns (fixed):** `version, algorithm, seed, game_id, episode, levels_completed, win_levels,
won, actions_used, invalid_actions, wall_seconds, timesteps_at_checkpoint, official_score,
checkpoint_path`.

**Why the three extra columns exist (added 25 Oct, see project-log.md for the full reasoning):**
* `timesteps_at_checkpoint` -- how much training the model had seen when this row was produced.
  Lets us plot levels/score against training progress, so we can see if a model is learning or
  just flat. `evaluate` receives it as an argument and stamps every row with it; it does not
  compute it itself.
* `official_score` -- the brief real scoring formula, `(human_actions / agent_actions) ** 2`
  (capped a little above 1), computed **only for won episodes**. Needs a human-actions baseline
  per game: use the `HUMAN_ACTIONS_PER_GAME` dict below (fill in a number per game id we actually
  evaluate, by playing the game at arcprize.org and counting your own actions). If a game has no
  entry yet, leave `official_score` as `NaN` rather than guessing.
* `checkpoint_path` -- the saved model file this row came from (same string `train_dqn`/`train_ppo`
  saves to), so any number in the report traces back to one reproducible file. `None` for the
  random policy, since it has no saved model.

**Protocol rules (write them in the markdown cell under the code, they go in the report):**
* evaluation always uses `HELD_OUT_GAME_IDS` unless stated;
* the policy acts deterministically during evaluation (for learned agents);
* the same `max_steps` cap for every version;
* **every** run is recorded, including ones that complete nothing.

**Done when:** the self-check prints `PASS` (it uses the stand-in `FakeArcEnv`, so you do not need Task 1).


In [ ]:


# ==============================================================================
# BLOCKER. Owner: Benny. Blocked by: nothing to start (build and self-check on FakeArcEnv).
# Needs Task 1 (Elias) only to evaluate on REAL games.
# Blocks: Task 7 runner (Elias) and Task 8 figures (Benny). The random_v0 baseline numbers come from here.
# ==============================================================================
# OWNER: Benny.  Only Benny edits this cell.

RESULT_COLUMNS = ["version", "algorithm", "seed", "game_id", "episode", "levels_completed",
                  "win_levels", "won", "actions_used", "invalid_actions", "wall_seconds",
                  "timesteps_at_checkpoint", "official_score", "checkpoint_path"]

HUMAN_ACTIONS_PER_GAME = {
    # "ar25-0c556536": 40,   # example: fill in with a real count once someone has played it
}


def official_score_for_episode(game_id, actions_used, won):
    """The brief real scoring formula: (human_actions / agent_actions) ** 2, capped a little
    above 1, computed only for WON episodes. Returns float('nan') if we have no human baseline
    for this game yet, or if the episode was not won (an unwon episode has no efficiency score).
    """
    if not won or game_id not in HUMAN_ACTIONS_PER_GAME:
        return float("nan")
    
    human_actions = HUMAN_ACTIONS_PER_GAME[game_id]
    return min((human_actions / actions_used) ** 2, 1.15)


def random_policy(observation, info, action_mask):
    """Pick uniformly among the allowed actions."""
    allowed_indices = np.flatnonzero(action_mask)
    return np.random.choice(allowed_indices)


def evaluate(policy_function, version, algorithm, seed, game_ids, episodes_per_game=1,
             max_steps=MAX_STEPS_PER_EPISODE, make_environment=None,
             timesteps_at_checkpoint=None, checkpoint_path=None):
    """Play every game in game_ids with policy_function and return one row per episode."""
    set_global_seed(seed)   # makes the policy's random choices repeatable for this seed
    if make_environment is None:
        make_environment = lambda ids, steps, s: ArcGymEnv(ids, max_steps=steps, seed=s)
        
    rows = []
    
    for game_id in game_ids:
        for episode in range(episodes_per_game):
            start_time = time.time()
            environment = make_environment([game_id], max_steps, seed)
            observation, info = environment.reset(seed=seed)
            
            invalid_actions = 0
            
            while True:
                action = policy_function(observation, info, environment.action_masks())
                observation, reward, terminated, truncated, info = environment.step(action)
                
                if not info.get("action_was_valid", True):
                    invalid_actions += 1
                    
                if terminated or truncated:
                    break
                    
            wall_seconds = time.time() - start_time
            won = "WIN" in str(info.get("state", ""))
            actions_used = info.get("actions_used", 0)
            score = official_score_for_episode(info.get("game_id", game_id), actions_used, won)
            
            rows.append({
                "version": version,
                "algorithm": algorithm,
                "seed": seed,
                "game_id": info.get("game_id", game_id),
                "episode": episode,
                "levels_completed": info.get("levels_completed", 0),
                "win_levels": info.get("win_levels", 0),
                "won": won,
                "actions_used": actions_used,
                "invalid_actions": invalid_actions,
                "wall_seconds": wall_seconds,
                "timesteps_at_checkpoint": timesteps_at_checkpoint,
                "official_score": score,
                "checkpoint_path": checkpoint_path
            })
            
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)


def save_results(results_table):
    """Append rows to RESULTS_CSV; write the header only if the file does not exist yet."""
    results_table.to_csv(RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False)


In [ ]:
# SELF-CHECK for Task 2 (do not edit)
def self_check_task2():
    try:
        table = evaluate(random_policy, "random_v0", "random", seed=0, game_ids=["fake-a", "fake-b"],
                         episodes_per_game=2, max_steps=30,
                         make_environment=lambda ids, steps, s: FakeArcEnv(ids, max_steps=steps, seed=s),
                         timesteps_at_checkpoint=0, checkpoint_path=None)
        assert list(table.columns) == RESULT_COLUMNS, "columns wrong"
        assert len(table) == 4, "expected 2 games x 2 episodes = 4 rows"
        assert (table["actions_used"] <= 30).all(), "max_steps not respected"
        assert (table["timesteps_at_checkpoint"] == 0).all(), "timesteps_at_checkpoint not stamped"
        assert table["official_score"].isna().all(), "fake games have no human baseline, score should be NaN"
        print(table)
        print("Task 2 PASS")
    except NotImplementedError as missing:
        print("Task 2 NOT DONE YET:", missing)

self_check_task2()


### Evaluation protocol (Benny writes this in plain words; it is frozen once Round 1 is approved)
- Games used for evaluation:
- Episodes per game:
- Action cap:
- Seeds:
- How a learned agent picks actions during evaluation:
- What is recorded:
- What counts as the official ARC score vs our local numbers:

In [ ]:
# NOTES FOR THE REPORT (Benny):
# -

## Task 3 (Paballo): DQN baseline  *(our course algorithm)*

> **BLOCKER FLAG:** BLOCKER. Blocked by: nothing to start (use FakeArcEnv); real training needs Task 1 (Elias). **Blocks: Task 7 runner (Elias)** for every dqn_* version.


**Why this matters:** one of the two required learning baselines. The report must explain **what is
learned** (the Q-network weights), **when** (every gradient step, from replayed past experience) and
**how it affects decisions** (greedy argmax, with epsilon-random exploration while training). Write
those three answers in your notes cell.

**You must deliver:**

| Function | Contract |
|---|---|
| `train_dqn(make_environment, seed, total_timesteps, version="dqn_v0_baseline") -> model` | trains SB3 `DQN("CnnPolicy", ...)` and saves it to `MODELS_DIR / f"{version}_seed{seed}.zip"` |
| `dqn_policy_function(model) -> policy_function` | returns a function with the same signature as `random_policy`, acting deterministically |

**Guidance:**
* Start from SB3's `DQN` (libraries are allowed by the brief). Our Lab 3 code is a good reference for
  explaining it, but the library version is quicker and less error-prone here.
* Suggested settings: `buffer_size=20_000`, `learning_starts=1_000`, `exploration_fraction=0.3`,
  `exploration_final_eps=0.05`, `target_update_interval=1_000`, `train_freq=4`. Write down any you change and why.
* DQN cannot use action masks. That is fine: invalid actions are no-ops in our environment. Note it as a
  known DQN disadvantage in the report.
* Wrap the environment in `stable_baselines3.common.monitor.Monitor` so episode rewards are logged.

**Done when:** the self-check prints `PASS` (uses `FakeArcEnv`).

In [ ]:
# ==============================================================================
# BLOCKER. Owner: Paballo. Blocked by: nothing to start; Task 1 (Elias) for real-game training.
# Blocks: Task 7 runner (Elias) for dqn_v0, dqn_v1, dqn_v2 versions.
# ==============================================================================
# OWNER: Paballo.  Only Paballo edits this cell.
import numpy as np
import torch
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback

def train_dqn(make_environment, seed, total_timesteps, version="dqn_v0_baseline"):
    """Train a DQN agent and save it using checkpoints."""
    set_global_seed(seed)
    environment = Monitor(make_environment(), filename=str(RESULTS_DIR / f"{version}_seed{seed}"))
    
    # Checkpoint setup to save progress periodically
    checkpoint_callback = CheckpointCallback(
        save_freq=10_000, 
        save_path=str(MODELS_DIR), 
        name_prefix=f"{version}_seed{seed}_cp"
    )
    
    model = DQN(
        "CnnPolicy", 
        environment, 
        seed=seed, 
        verbose=1,
        buffer_size=20_000,
        learning_starts=1_000,
        exploration_fraction=0.3,
        exploration_final_eps=0.05,
        target_update_interval=1_000,
        train_freq=4
    )
    
    model.learn(total_timesteps=total_timesteps, callback=checkpoint_callback)
    model.save(MODELS_DIR / f"{version}_seed{seed}")
    return model


def dqn_policy_function(model):
    """Return policy_function(observation, info, action_mask) -> int for evaluate().
    Plain DQN baseline: ignores action_mask on purpose (DQN has no masking); the adapter
    turns invalid presses into no-ops and evaluate counts them as invalid_actions."""
    def policy_function(observation, info, action_mask):
        action, _ = model.predict(observation, deterministic=True)  # deterministic = always its best guess
        return int(action)
    return policy_function


In [ ]:
# SELF-CHECK for Task 3 (do not edit the PASS checks; the report section below only prints)
import time
import pandas as pd

def self_check_task3():
    try:
        # ---- 1. Train a tiny model (same as before) and time it ----
        start = time.time()
        model = train_dqn(lambda: FakeArcEnv(max_steps=50), seed=0,
                          total_timesteps=TRAIN_TIMESTEPS_SMOKE, version="selfcheck_dqn")
        train_seconds = time.time() - start

        # ---- 2. The original PASS checks (unchanged) ----
        policy = dqn_policy_function(model)
        test_environment = FakeArcEnv(max_steps=10)
        observation, info = test_environment.reset()
        action = policy(observation, info, test_environment.action_masks())
        assert isinstance(action, int) and 0 <= action < N_ACTIONS
        assert (MODELS_DIR / "selfcheck_dqn_seed0.zip").exists(), "model was not saved"

        # ---- 3. Reviewer report: is it set up right, and is training doing anything? ----
        print("=== Settings actually used (these go in the report) ===")
        print(f"buffer_size={model.buffer_size}  learning_starts={model.learning_starts}  "
              f"train_freq={model.train_freq.frequency}  target_update_interval={model.target_update_interval}")
        print(f"exploration: start={model.exploration_initial_eps}  end={model.exploration_final_eps}  "
              f"fraction={model.exploration_fraction}  -> rate now={model.exploration_rate:.3f}")
        print(f"gamma={model.gamma}  learning_rate={model.learning_rate}  batch_size={model.batch_size}")

        print("\n=== Training run ===")
        monitor = model.get_env().envs[0]                    # the Monitor wrapper train_dqn added
        episode_rewards = monitor.get_episode_rewards()      # true reward per finished episode
        episode_lengths = monitor.get_episode_lengths()
        gradient_updates = max(0, (model.num_timesteps - model.learning_starts) // model.train_freq.frequency)
        print(f"timesteps={model.num_timesteps}  wall time={train_seconds:.1f}s  "
              f"episodes finished={len(episode_rewards)}  approx gradient updates={gradient_updates}")
        if episode_rewards:
            half = len(episode_rewards) // 2 or 1
            print(f"mean reward, first half of episodes={np.mean(episode_rewards[:half]):.3f}  "
                  f"second half={np.mean(episode_rewards[half:]):.3f}  (second > first hints at learning)")
            print(f"mean episode length={np.mean(episode_lengths):.1f}  max reward in one episode={max(episode_rewards)}")

        print("\n=== Saved files ===")
        for path in sorted(MODELS_DIR.glob("selfcheck_dqn*")):
            print("  ", path.name)

        print("\n=== Greedy behaviour after training ===")
        actions_taken = []
        observation, info = test_environment.reset()
        for _ in range(10):
            actions_taken.append(policy(observation, info, test_environment.action_masks()))
            observation, _, terminated, truncated, info = test_environment.step(actions_taken[-1])
            if terminated or truncated:
                break
        print(f"actions picked: {actions_taken}  distinct={len(set(actions_taken))}  "
              f"(1 distinct = the policy has collapsed onto a single button)")

        print("\n=== Trained DQN vs random, same fake games, 5 seeds ===")
        fake_maker = lambda ids, steps, s: FakeArcEnv(ids, max_steps=steps, seed=s)
        comparison = []
        for name, chosen_policy in [("dqn", policy), ("random", random_policy)]:
            for evaluation_seed in range(5):
                table = evaluate(chosen_policy, f"selfcheck_{name}", name, evaluation_seed, ["fake-x"],
                                 max_steps=50, make_environment=fake_maker)
                comparison.append(table)
        comparison = pd.concat(comparison)
        print(comparison.groupby("algorithm")[["levels_completed", "actions_used", "invalid_actions"]].mean())
        print("(At 2,000 smoke steps DQN is NOT expected to beat random. This only proves the pipeline runs.)")

        print("\nTask 3 PASS")
    except NotImplementedError as missing:
        print("Task 3 NOT DONE YET:", missing)

self_check_task3()


In [ ]:
# NOTES FOR THE REPORT (Paballo): what is learned / when / how it affects decisions; settings changed; AI help.
# -

## Task 4 (Benny): PPO baseline with action masking  *(our non-course algorithm)*

> **BLOCKER FLAG:** BLOCKER. Blocked by: nothing to start (use FakeArcEnv); real training needs Task 1 (Elias). **Blocks: Task 7 runner (Elias)** for every ppo_* version.


**Why this matters:** the brief asks for more depth on the unfamiliar algorithm: its objective (the
clipped surrogate), what is learned (policy and value networks), when (alternating collect-then-update
phases, on-policy), how it affects decisions (sampling from the policy, masked to legal actions).

**You must deliver:**

| Function | Contract |
|---|---|
| `train_ppo(make_environment, seed, total_timesteps, version="ppo_v0_baseline") -> model` | trains sb3-contrib `MaskablePPO("CnnPolicy", ...)`, saves to `MODELS_DIR` |
| `ppo_policy_function(model) -> policy_function` | deterministic, **uses** `action_mask` |

**Guidance:** `n_steps=512`, `batch_size=64`, `learning_rate=3e-4`, `ent_coef=0.01` (a little entropy
bonus helps exploration when reward is this sparse). `MaskablePPO` reads masks by calling the
environment's `action_masks()` method. Any wrapper we add later must pass that method through.

In [ ]:
# ==============================================================================
# BLOCKER. Owner: Benny. Blocked by: nothing to start; Task 1 (Elias) for real-game training.
# Blocks: Task 7 runner (Elias) for ppo_v0, ppo_v1, ppo_v2 versions.
# ==============================================================================
# OWNER: Benny.  Only Benny edits this cell.
from sb3_contrib import MaskablePPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback

def train_ppo(make_environment, seed, total_timesteps, version="ppo_v0_baseline"):
    """Train a PPO agent and save it using checkpoints."""
    set_global_seed(seed)
    environment = Monitor(make_environment(), filename=str(RESULTS_DIR / f"{version}_seed{seed}"))
    
    # Checkpoint setup to save progress periodically
    checkpoint_callback = CheckpointCallback(
        save_freq=10_000, 
        save_path=str(MODELS_DIR), 
        name_prefix=f"{version}_seed{seed}_cp"
    )
    
    model = MaskablePPO(
        "CnnPolicy", 
        environment, 
        seed=seed,
        n_steps=512,
        batch_size=64,
        learning_rate=3e-4,
        ent_coef=0.01,
        verbose=1
    )
    
    model.learn(total_timesteps=total_timesteps, callback=checkpoint_callback)
    model.save(MODELS_DIR / f"{version}_seed{seed}")
    return model


def ppo_policy_function(model):
    """Return policy_function(observation, info, action_mask) -> int or array for evaluate()."""
    def policy_function(observation, info, action_mask):
        action, _ = model.predict(observation, action_masks=action_mask, deterministic=True)
        # If action is an array (like [move, slot] for Adaptive Stacking), return it as-is.
        # Otherwise, safely cast it to an integer for standard actions.
        if isinstance(action, np.ndarray) and action.size > 1:
            return action
        return int(action)
    return policy_function

In [ ]:
# SELF-CHECK for Task 4 (PASS checks unchanged; the lines after them only print)
def self_check_task4():
    try:
        model = train_ppo(lambda: FakeArcEnv(max_steps=50), seed=0,
                          total_timesteps=TRAIN_TIMESTEPS_SMOKE, version="selfcheck_ppo")
        policy = ppo_policy_function(model)
        test_environment = FakeArcEnv(max_steps=10)
        observation, info = test_environment.reset()
        action = policy(observation, info, test_environment.action_masks())
        assert isinstance(action, int) and 0 <= action < N_ACTIONS

        rewards = model.get_env().envs[0].get_episode_rewards()
        half = len(rewards) // 2 or 1
        print(f"settings: n_steps={model.n_steps} batch={model.batch_size} lr={model.learning_rate} ent_coef={model.ent_coef}")
        print(f"episodes={len(rewards)}  mean reward 1st half={np.mean(rewards[:half]):.2f}  2nd half={np.mean(rewards[half:]):.2f}")
        actions = [policy(test_environment.reset()[0], {}, test_environment.action_masks()) for _ in range(5)]
        print(f"greedy actions on 5 fresh screens: {actions} (all the same = collapsed)")
        fake_maker = lambda ids, steps, s: FakeArcEnv(ids, max_steps=steps, seed=s)
        for name, chosen in [("ppo", policy), ("random", random_policy)]:
            levels = [evaluate(chosen, name, name, s, ["fake-x"], max_steps=50, make_environment=fake_maker)["levels_completed"].iloc[0] for s in range(5)]
            print(f"{name}: mean levels over 5 seeds = {np.mean(levels):.1f}")
        print("Task 4 PASS")
    except NotImplementedError as missing:
        print("Task 4 NOT DONE YET:", missing)

self_check_task4()


In [ ]:
# NOTES FOR THE REPORT (Benny):
# -

---
# Part 4: Workspace, Round 2 (improvements, experiments, packaging)

Start Round 2 only when your Round 1 task is APPROVED in the review log.
For each improvement the report needs four sentences: **observed limitation, hypothesis,
intervention, what the results show.** Write them in your notes cell as you go.

## Task 5 (Paballo): count-based novelty bonus wrapper

> **BLOCKER FLAG:** Blocked by: Paballo's Task 3 being approved first (team rule). Technically buildable on FakeArcEnv now. **Blocks: Task 7 runner (Elias)** for the *_v2_memory_novelty and *_v2_novelty_only versions.


**Limitation it targets:** reward is zero until a level is finished, so nothing is learned until the
agent finishes one by accident. **Hypothesis:** paying a small bonus for screens the agent has rarely
seen makes it explore systematically and finish its first level sooner.

**You must deliver** `NoveltyBonus(gym.Wrapper)`:
* `__init__(self, environment, bonus_scale=0.1)`
* keeps a dictionary `visit_counts` from a hash of the observation bytes to how often it was seen
  (`hashlib.md5(observation.tobytes()).hexdigest()`)
* `step` returns `reward + bonus_scale / sqrt(count)` and adds `info["true_reward"]` (the unshaped
  reward) and `info["novelty_bonus"]`
* `action_masks(self)` that returns `self.env.action_masks()` (**required** or MaskablePPO breaks)
* counts are **not** cleared on reset (novelty is measured across the whole training run)

**Reminder:** the bonus is training-only. Evaluation always reports the true reward / levels.

In [ ]:
# ==============================================================================
# Owner: Paballo. Blocked by: Task 3 approval (team rule); no code dependency, works on FakeArcEnv.
# Blocks: Task 7 runner (Elias) for the v2 and novelty-only ablation versions.
# ==============================================================================
# OWNER: Paballo.
import hashlib
import numpy as np
import gymnasium as gym

class NoveltyBonus(gym.Wrapper):
    """Adds bonus_scale / sqrt(visit count) to the reward for each observation."""

    def __init__(self, environment, bonus_scale=0.1):
        super().__init__(environment)
        self.bonus_scale = bonus_scale
        self.visit_counts = {}

    def step(self, action):
        observation, reward, terminated, truncated, info = self.env.step(action)
        
        obs_hash = hashlib.md5(observation.tobytes()).hexdigest()
        
        self.visit_counts[obs_hash] = self.visit_counts.get(obs_hash, 0) + 1
        
        bonus = self.bonus_scale / np.sqrt(self.visit_counts[obs_hash])
        
        info["true_reward"] = reward
        info["novelty_bonus"] = bonus
        
        return observation, reward + bonus, terminated, truncated, info

    def action_masks(self):
        return self.env.action_masks()

In [ ]:
# SELF-CHECK for Task 5 (PASS checks unchanged; the lines after them only print)
def self_check_task5():
    try:
        environment = NoveltyBonus(FakeArcEnv(max_steps=20), bonus_scale=0.1)
        environment.reset()
        _, first_reward, _, _, first_info = environment.step(0)
        _, second_reward, _, _, second_info = environment.step(0)
        assert "true_reward" in first_info and "novelty_bonus" in first_info
        assert second_info["novelty_bonus"] < first_info["novelty_bonus"], "repeat screens should earn less"
        assert environment.action_masks().shape == (N_ACTIONS,)

        print(f"same screen twice: bonus {first_info['novelty_bonus']:.3f} -> {second_info['novelty_bonus']:.3f} (should shrink)")
        print(f"reward = true + bonus? {abs(second_reward - (second_info['true_reward'] + second_info['novelty_bonus'])) < 1e-9}")

        counts_before_reset = len(environment.visit_counts)
        environment.reset()
        print(f"screens remembered across reset: {len(environment.visit_counts)} (should equal {counts_before_reset}, not 0)")

        total_bonus, total_true = 0.0, 0.0
        for _ in range(200):
            _, _, terminated, truncated, info = environment.step(int(np.random.randint(N_ACTIONS)))
            total_bonus += info["novelty_bonus"]; total_true += info["true_reward"]
            if terminated or truncated:
                environment.reset()
        print(f"200 random steps: distinct screens={len(environment.visit_counts)}  "
              f"total bonus={total_bonus:.2f}  total true reward={total_true:.0f}")
        print("(true reward must never include the bonus: it's what evaluation reports)")
        print("Task 5 PASS")
    except NotImplementedError as missing:
        print("Task 5 NOT DONE YET:", missing)

self_check_task5()


In [ ]:
# NOTES FOR THE REPORT (Paballo): limitation, hypothesis, intervention, result.
# -

## Task 6 (Benny): memory wrappers (frame stacking, then Adaptive Stacking)

> **BLOCKER FLAG:** Blocked by: Benny's Task 4 being approved first (team rule). Technically buildable on FakeArcEnv now. **Blocks: Task 7 runner (Elias)** for every *_v1_memory and *_v2_memory_novelty version, and **Task 8 Kaggle agent (Benny)** if the final submitted agent uses memory.


**Limitation it targets:** many games hide state off-screen, so one screen is not enough to decide.
**Hypothesis:** giving the agent several past screens fixes this; letting it **choose** which screens
to keep (Adaptive Stacking, Nangue Tasse et al.) does better than keeping the last few.

**You must deliver:**
* `FrameStackImage(environment, k)`: observation becomes `(k, 64, 64)`, oldest screen dropped first.
* `AdaptiveStackImage(environment, k, flatten_for_dqn=False)`: same observation shape; the agent
  additionally picks which of the k slots to overwrite each step.
  * for PPO: `action_space = MultiDiscrete([70, k])`, and `action_masks()` returns the 70-long game mask
    concatenated with k `True` values;
  * for DQN (`flatten_for_dqn=True`): `action_space = Discrete(70 * k)`, decoded with `divmod(a, k)`.
* both wrappers pass `action_masks()` through.

Port the logic from the Research Project's `src/memory/adaptive_stack.py`; the only change is that one
slot now holds a 64 by 64 screen instead of a single number.

In [ ]:
# ==============================================================================
# Owner: Benny. Blocked by: Task 4 approval (team rule); no code dependency, works on FakeArcEnv.
# Blocks: Task 7 runner (Elias) for v1 and v2 versions; Task 8 Kaggle agent (Benny) if the final agent uses memory.
# ==============================================================================
# OWNER: Benny.
from gymnasium import spaces
from collections import deque
class FrameStackImage(gym.Wrapper):
    """Stacks the last k frames to provide short-term memory."""
    def __init__(self, environment, k):
        super().__init__(environment)
        self.k = k
        self.frames = deque(maxlen=k)
        
        orig_space = environment.observation_space
        self.observation_space = spaces.Box(
            low=orig_space.low.min(),
            high=orig_space.high.max(),
            shape=(k, 64, 64),
            dtype=orig_space.dtype
        )

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for _ in range(self.k):
            self.frames.append(obs[0])
        return np.stack(self.frames, axis=0), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.frames.append(obs[0])
        return np.stack(self.frames, axis=0), reward, terminated, truncated, info

    def action_masks(self):
        return self.env.action_masks()


class AdaptiveStackImage(gym.Wrapper):
    """Allows the agent to choose which of the k memory slots to overwrite."""
    def __init__(self, environment, k, flatten_for_dqn=False):
        super().__init__(environment)
        self.k = k
        self.flatten_for_dqn = flatten_for_dqn
        
        orig_space = environment.observation_space
        self.observation_space = spaces.Box(
            low=orig_space.low.min(),
            high=orig_space.high.max(),
            shape=(k, 64, 64),
            dtype=orig_space.dtype
        )
        
        if self.flatten_for_dqn:
            self.action_space = spaces.Discrete(70 * k)
        else:
            self.action_space = spaces.MultiDiscrete([70, k])
            
        self.frames = np.zeros((k, 64, 64), dtype=orig_space.dtype)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for i in range(self.k):
            self.frames[i] = obs[0]
        return self.frames.copy(), info

    def step(self, action):
        if self.flatten_for_dqn:
            game_action, memory_slot = divmod(action, self.k)
        else:
            game_action, memory_slot = action[0], action[1]
            
        obs, reward, terminated, truncated, info = self.env.step(game_action)
        self.frames[memory_slot] = obs[0]
        
        return self.frames.copy(), reward, terminated, truncated, info

    def action_masks(self):
        game_mask = self.env.action_masks()
        if self.flatten_for_dqn:
            return np.repeat(game_mask, self.k)
        else:
            return np.concatenate([game_mask, np.ones(self.k, dtype=bool)])

In [ ]:
# SELF-CHECK for Task 6 (PASS checks unchanged; the lines after them only print)
def self_check_task6():
    try:
        stacked = FrameStackImage(FakeArcEnv(max_steps=10), k=4)
        observation, _ = stacked.reset()
        assert observation.shape == (4, 64, 64)
        adaptive = AdaptiveStackImage(FakeArcEnv(max_steps=10), k=4)
        observation, _ = adaptive.reset()
        assert observation.shape == (4, 64, 64)
        assert adaptive.action_masks().shape == (N_ACTIONS + 4,)
        adaptive.step(np.array([0, 2]))
        flat = AdaptiveStackImage(FakeArcEnv(max_steps=10), k=4, flatten_for_dqn=True)
        flat.reset()
        assert flat.action_space.n == N_ACTIONS * 4
        flat.step(5)

        # Frame stack: after a step, the NEWEST screen must sit in the last slot
        observation, _ = stacked.reset()
        target = N_SIMPLE_ACTIONS + stacked.env.target_block      # the fake game's winning click
        new_observation, *_ = stacked.step(target)                # target moves, so the screen changes
        print(f"frame stack: last slot changed after a step = {not np.array_equal(new_observation[-1], observation[-1])}, "
              f"first 3 slots unchanged = {np.array_equal(new_observation[:3], observation[1:])}")

        # Adaptive: only the CHOSEN slot should change
        observation, _ = adaptive.reset()
        target = N_SIMPLE_ACTIONS + adaptive.env.target_block
        new_observation, *_ = adaptive.step(np.array([target, 2]))
        changed_slots = [i for i in range(4) if not np.array_equal(new_observation[i], observation[i])]
        print(f"adaptive: slots that changed = {changed_slots} (should be [2])")

        # Flat DQN decoding: option 41 with k=4 should mean move 10, slot 1
        print(f"flat decode of 41: {divmod(41, 4)} (should be (10, 1))  flat mask length = {flat.action_masks().shape[0]} (should be {N_ACTIONS * 4})")
        print("Task 6 PASS")
    except NotImplementedError as missing:
        print("Task 6 NOT DONE YET:", missing)

self_check_task6()


In [ ]:
# NOTES FOR THE REPORT (Benny): limitation, hypothesis, intervention, result.
# -

## Task 7 (Elias): the experiment runner

> **BLOCKER FLAG:** BLOCKED. Needs: Task 1 (Elias), Task 2 (Benny), Task 3 (Paballo), Task 4 (Benny) to run the baselines; Task 5 (Paballo) and Task 6 (Benny) for the improved versions. **Blocks: all experiment results**, so Task 8 figures (Benny) and the report.


**You must deliver** `run_version(version, algorithm, build_training_environment, seeds=SEEDS,
total_timesteps=TRAIN_TIMESTEPS_REAL)` that, for each seed: trains (with `train_dqn` or `train_ppo`),
evaluates on `HELD_OUT_GAME_IDS` with `evaluate`, saves with `save_results`, and **skips seeds already
in `RESULTS_CSV`** (Colab sessions die; a rerun must resume, not restart).

Also deliver a dictionary `VERSIONS` mapping each version name in Part 1 to its algorithm and its
environment builder (for example `lambda: NoveltyBonus(FrameStackImage(ArcGymEnv(TRAINING_GAME_IDS), 4))`).
Note: an AS-wrapped environment changes the observation the policy sees, so evaluation of that version
must use the same wrappers. Build `make_environment` for `evaluate` from the same builder.

In [ ]:
# ==============================================================================
# BLOCKED. Owner: Elias. Blocked by: Tasks 1, 2, 3, 4 for baselines; Tasks 5 and 6 for v1/v2 versions.
# Can be written now against the function names in the contracts, then run once they pass.
# Blocks: every results row, so Task 8 figures (Benny) and the whole report.
# ==============================================================================
# OWNER: Elias.
import pandas as pd

def run_version(version, algorithm, build_training_environment, seeds=SEEDS,
                total_timesteps=TRAIN_TIMESTEPS_REAL):
    """Run training and evaluation for a specific version across multiple seeds."""
    completed_seeds = set()
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        if not df.empty and "version" in df.columns and "seed" in df.columns:
            completed_seeds = set(df[df["version"] == version]["seed"].unique())

    for seed in seeds:
        if seed in completed_seeds:
            print(f"Skipping {version} seed {seed}, already found in results.")
            continue
            
        print(f"Running {version} seed {seed}...")
        
        if algorithm == "dqn":
            model = train_dqn(build_training_environment, seed, total_timesteps, version)
            policy = dqn_policy_function(model)
        elif algorithm == "ppo":
            model = train_ppo(build_training_environment, seed, total_timesteps, version)
            policy = ppo_policy_function(model)
        else:
            raise ValueError(f"Unknown algorithm: {algorithm}")
            
        def make_eval_env(ids, steps, s):
            env = build_training_environment()
            env.unwrapped.game_ids = list(ids)
            env.unwrapped.max_steps = steps
            return env
            
        results_df = evaluate(
            policy_function=policy,
            version=version,
            algorithm=algorithm,
            seed=seed,
            game_ids=HELD_OUT_GAME_IDS,
            make_environment=make_eval_env,
            timesteps_at_checkpoint=total_timesteps,
            checkpoint_path=str(MODELS_DIR / f"{version}_seed{seed}")
        )
        
        save_results(results_df)


# Sanity check: evaluate the completely random agent before running the smart bots
# for seed in SEEDS:
#     save_results(evaluate(random_policy, "random_v0", "random", seed, HELD_OUT_GAME_IDS, timesteps_at_checkpoint=0))


VERSIONS = {
    # v0: plain agents
    "dqn_v0_baseline": {"algorithm": "dqn", "build_training_environment": lambda: ArcGymEnv(TRAINING_GAME_IDS)},
    "ppo_v0_baseline": {"algorithm": "ppo", "build_training_environment": lambda: ArcGymEnv(TRAINING_GAME_IDS)},
    
    # v1: + memory (frame stacking)
    "dqn_v1_memory": {"algorithm": "dqn", "build_training_environment": lambda: FrameStackImage(ArcGymEnv(TRAINING_GAME_IDS), 4)},
    "ppo_v1_memory": {"algorithm": "ppo", "build_training_environment": lambda: FrameStackImage(ArcGymEnv(TRAINING_GAME_IDS), 4)},
    
    # v2: v1 + novelty. Novelty sits INSIDE the stack so it counts single screens, not 4-screen sequences
    "dqn_v2_memory_novelty": {"algorithm": "dqn", "build_training_environment": lambda: FrameStackImage(NoveltyBonus(ArcGymEnv(TRAINING_GAME_IDS)), 4)},
    "ppo_v2_memory_novelty": {"algorithm": "ppo", "build_training_environment": lambda: FrameStackImage(NoveltyBonus(ArcGymEnv(TRAINING_GAME_IDS)), 4)},
    
    # ablation: novelty WITHOUT memory, isolates what novelty does on its own
    "dqn_v2_novelty_only": {"algorithm": "dqn", "build_training_environment": lambda: NoveltyBonus(ArcGymEnv(TRAINING_GAME_IDS))},
    "ppo_v2_novelty_only": {"algorithm": "ppo", "build_training_environment": lambda: NoveltyBonus(ArcGymEnv(TRAINING_GAME_IDS))},
    
    # optional extra, only if time: Adaptive Stacking instead of frame stacking
    "dqn_v1_adaptive": {"algorithm": "dqn", "build_training_environment": lambda: AdaptiveStackImage(ArcGymEnv(TRAINING_GAME_IDS), 4, flatten_for_dqn=True)},
    "ppo_v1_adaptive": {"algorithm": "ppo", "build_training_environment": lambda: AdaptiveStackImage(ArcGymEnv(TRAINING_GAME_IDS), 4)},
}

In [ ]:
# The actual training 
## this where we train 
import time
for name, spec in VERSIONS.items():
    if "adaptive" in name:          # optional Adaptive Stacking versions: skipped unless there's time
        continue
    start = time.time()
    run_version(name, spec["algorithm"], spec["build_training_environment"])
    print(f"done {name}: all seeds in {(time.time() - start) / 60:.1f} min")

## Task 8 (Benny): figures, tables, and the Kaggle agent

> **BLOCKER FLAG:** Partly blocked. **Random-agent Kaggle submission (due 20 Oct): blocked by nothing, do it early.** Figures: blocked by Task 7 results (can be built now on the fake rows from the Task 2 self-check). Trained Kaggle agent: blocked by Task 1 (Elias, frame_to_observation) and a chosen trained model from Task 7. Blocks: the report's results section and the final submission.

.ables (from `RESULTS_CSV`):**
1. Bar chart: mean levels completed per version on held-out games, with each seed shown as a dot.
2. Table: per version, mean and standard deviation of levels completed, games won, actions used,
   invalid actions, training wall time.
3. Learning curves: episode reward against timesteps from the Monitor logs, one line per seed.
4. Ablation table: `v2_memory_novelty` vs `v1_memory` vs `v2_novelty_only`, per algorithm.

**Kaggle agent:** a class `MyAgent` in the shape the official starter kit expects
(`is_done(self, frames, latest_frame) -> bool` and `choose_action(self, frames, latest_frame) -> GameAction`),
which loads our chosen saved model and converts `latest_frame` to our observation exactly as
`ArcGymEnv.frame_to_observation` does. Start from the starter kit: `github.com/arcprize/ARC-AGI-3-Kaggle-Starter`
(`agent/my_agent.py`). **A valid submission is worth more than a clever one**: submit the random agent
first (by 20 Oct) to prove the pipeline, then swap in the trained model.

In [ ]:
# ==============================================================================
# Owner: Benny. Random-agent Kaggle submission: blocked by NOTHING (do it by 20 Oct).
# Figures/tables: blocked by Task 7 results (build now on fake rows). Trained Kaggle agent: blocked by
# Task 1 (Elias) and a model from Task 7. Blocks: report results section and final submission.
# ==============================================================================
# OWNER: Benny.
import matplotlib.pyplot as plt

import pandas as pd
import matplotlib.pyplot as plt

def make_figures_and_tables(results_csv=RESULTS_CSV):
    """Generates the required charts and tables from RESULTS_CSV for the final report."""
    if not results_csv.exists():
        print("No results to plot yet. Run the experiments first!")
        return

    df = pd.read_csv(results_csv)
    df = df[~df["version"].str.startswith("selfcheck")]   # drop smoke-test rows
    
    # 1. Bar chart: mean levels completed per version, with seeds as dots
    plt.figure(figsize=(10, 6))
    means = df.groupby("version")["levels_completed"].mean()
    plt.bar(means.index, means.values, alpha=0.7, color='skyblue', edgecolor='black')
    
    # Add dots for individual seeds
    for version in means.index:
        seed_data = df[df["version"] == version].groupby("seed")["levels_completed"].mean()
        x_pos = means.index.get_loc(version)
        plt.scatter([x_pos]*len(seed_data), seed_data.values, color='red', zorder=5)
        
    plt.title("Average Levels Beaten per Bot Version")
    plt.ylabel("Levels Completed")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "levels_completed_bar_chart.png")
    plt.close()

    # 2. Table: per version stats (means and standard deviations)
    stats_table = df.groupby("version").agg({
        "levels_completed": ["mean", "std"],
        "won": ["mean"],
        "actions_used": ["mean", "std"],
        "invalid_actions": ["mean", "std"],
        "wall_seconds": ["mean"]
    }).round(2)
    seed_spread = df.groupby(["version", "seed"])["levels_completed"].mean().groupby("version").agg(["mean", "std"])
    print("\n--- Spread across seeds ---"); print(seed_spread); seed_spread.to_csv(RESULTS_DIR / "seed_spread.csv")

    
    print("\n--- Performance Stats Table ---")
    print(stats_table)
    stats_table.to_csv(RESULTS_DIR / "stats_table.csv")

    # 3. Ablation table: Isolating the upgrades
    ablation_versions = ["dqn_v1_memory", "dqn_v2_novelty_only", "dqn_v2_memory_novelty",
                         "ppo_v1_memory", "ppo_v2_novelty_only", "ppo_v2_memory_novelty"]
    ablation_df = df[df["version"].isin(ablation_versions)]
    
    if not ablation_df.empty:
        ablation_table = ablation_df.groupby(["algorithm", "version"])["levels_completed"].mean().reset_index()
        print("\n--- Ablation Table ---")
        print(ablation_table)
        ablation_table.to_csv(RESULTS_DIR / "ablation_table.csv")
    # 3. Learning curves: training reward against training steps, one line per seed
    versions = sorted({f.name.split("_seed")[0] for f in RESULTS_DIR.glob("*_seed*.monitor.csv") if not f.name.startswith("selfcheck")})
    for version in versions:
        plt.figure(figsize=(8, 4))
        for monitor_file in sorted(RESULTS_DIR.glob(f"{version}_seed*.monitor.csv")):
            log = pd.read_csv(monitor_file, skiprows=1)          # first line of a monitor file is a header comment
            plt.plot(log["l"].cumsum(), log["r"].rolling(20, min_periods=1).mean(), label=monitor_file.name.split(".")[0][-5:])
        shaped = " (includes novelty bonus)" if "novelty" in version else ""
        plt.title(f"{version}: training reward{shaped}"); plt.xlabel("training steps"); plt.ylabel("episode reward (rolling mean of 20)")
        plt.legend(); plt.tight_layout(); plt.savefig(RESULTS_DIR / f"learning_curve_{version}.png"); plt.close()

class MyAgent:
    """Kaggle submission agent. Put this inside agent/my_agent.py for the official starter kit."""
    
    def __init__(self, model_path, algorithm="ppo"):
        # Load up the saved brain file depending on which bot we are submitting
        if algorithm == "ppo":
            from sb3_contrib import MaskablePPO
            self.model = MaskablePPO.load(model_path)
        else:
            from stable_baselines3 import DQN
            self.model = DQN.load(model_path)

    def is_done(self, frames, latest_frame):
        """Tells the Kaggle runner if we just died or beat the whole game."""
        state = getattr(latest_frame, "state", "UNKNOWN")
        return state in ["WIN", "GAME_OVER"]

    def _frame_to_observation(self, latest_frame):
        """Exact copy of our screen resizer so the raw Kaggle game looks right to the bot."""
        last_grid = latest_frame.frame[-1]
        grid_array = np.array(last_grid, dtype=np.int32)
        
        h, w = grid_array.shape
        pad_h = max(0, 64 - h)
        pad_w = max(0, 64 - w)
        padded_array = np.pad(grid_array, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0)
        
        scaled_array = (padded_array * 16).astype(np.uint8)
        return np.expand_dims(scaled_array, axis=0)

    def choose_action(self, frames, latest_frame):
        """Looks at the screen and presses the best button for the Kaggle test."""
        obs = self._frame_to_observation(latest_frame)
        
        # We have to rebuild the button mask manually here
        mask = np.zeros(70, dtype=bool)
        available = getattr(latest_frame, "available_actions", [])
        
        for i, act in enumerate([1, 2, 3, 4, 5, 7]):
            if act in available: mask[i] = True
        if 6 in available: mask[6:] = True
        
        # Ask the bot's brain what to do
        action, _ = self.model.predict(obs, action_masks=mask, deterministic=True)
        action_index = int(action)
        
        # Translate the math number back into a real game move
        simple_actions = [1, 2, 3, 4, 5, 7]
        if action_index < 6:
            return simple_actions[action_index], None
        else:
            click_idx = action_index - 6
            return 6, {"x": click_idx % 64, "y": click_idx // 64}



In [ ]:
make_figures_and_tables()

---
# Part 5: Review log and hand-off (Benny maintains)

| Task | Owner | Self-check | Review status | Review notes | In `arc_project.py`? |
|---|---|---|---|---|---|
| 0 Interface probe | Elias | n/a | | | n/a |
| 1 ArcGymEnv | Elias | | | | |
| 2 Evaluation | Benny | | | | |
| 3 DQN baseline | Paballo | | | | |
| 4 PPO baseline | Benny | | | | |
| 5 Novelty bonus | Paballo | | | | |
| 6 Memory wrappers | Benny | | | | |
| 7 Runner | Elias | | | | |
| 8 Figures + Kaggle agent | Benny | | | | |

Status values: `NOT STARTED`, `IN PROGRESS`, `PASS (awaiting review)`, `CHANGES REQUESTED`, `APPROVED`.

### What happens after everything is APPROVED
1. The approved cells are aggregated into one file, `arc_project.py`, with a command line:
   `python arc_project.py --version ppo_v1_memory --seeds 0 1 2`. That is what we run for the real
   experiments (Colab or any machine).
2. The chosen final agent becomes `agent/my_agent.py` in the Kaggle starter kit, which builds the
   submission notebook. That notebook is downloaded for the Moodle ZIP.
3. The report is written from the notes cells and the figures, in the RLC 2026 template (8 pages,
   cover page disabled).
4. The Moodle ZIP (`<Kaggle-team-name>.zip`): report PDF, Kaggle notebook, competition evidence
   (submission id, score, timestamp, nominated entry, Paper Track link and a saved copy), and one
   sentence per member on who did what.

### Contribution statement (fill in as you go)
- Banele Mkhize:
- Elias Sepuru:
- Paballo:
- Banele Mkhize: